# SLS saved 100 epoch results

Inspect completed synthetic experiments, without rerunning the solver.
查看已完成的 100 轮结果；运行本 Notebook 不会调用正演、编译、安装或联网。

The original executed GPU notebook covered five epochs. These 100-epoch arrays were written by separate runs of the same workflow. This is a new viewer, not a newly executed GPU notebook.


In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
ROOT = Path.cwd()
assert (ROOT / "data" / "joint_Vp_Q" / "summary.json").is_file(), "Open this notebook from the extracted example directory."


## Verify whole grid and trainable region RMSE

Saved float32 history values may differ slightly from the float64 recomputation below.


In [ ]:
for name in ["fixed_Q_Vp", "joint_Vp_Q"]:
    p = ROOT / "data" / name
    summary = json.loads((p / "summary.json").read_text())
    initial = np.load(p / "models_initial.npz", allow_pickle=False)
    final = np.load(p / "models_epoch_100.npz", allow_pickle=False)
    mask = initial["gradient_mask"].astype(bool)
    print(name, summary["status"], summary["completed_epochs"], "epochs")
    print("Full-survey normalized MSE:", summary["initial_normalized_MSE"], "->", summary["final_normalized_MSE"])
    for key, truth in [("vp", "vp_true"), ("Q", "q_true")]:
        error = final[key].astype(float) - initial[truth].astype(float)
        print(key, "whole RMSE", np.sqrt(np.mean(error**2)), "trainable RMSE", np.sqrt(np.mean(error[mask]**2)))


## Models and recorded training histories

The fixed-Q panel uses true Q throughout and is not Q recovery. Fixed water Q=1000 lies above the displayed rock-Q color scale.


In [ ]:
display(Image(filename=str(ROOT / "figures" / "marmousi100_models_comparison.png")))


## Why the loss curve is insufficient for Q recovery

Joint Q RMSE is lowest at epoch 59 (70.131958) and increases to 82.819542 by epoch 100. The figure marker at epoch 60 identifies the nearest saved model. Training losses average batches evaluated before different updates; full-survey evaluations use one fixed model. True-Q RMSE is only available in this synthetic experiment, not as a field-data stopping criterion.

Vp-Q trade-off is plausible, but no matched Marmousi Q-only or sensitivity/learning-rate ablation isolates its cause.


In [ ]:
display(Image(filename=str(ROOT / "figures" / "joint_loss_and_model_errors.png")))


In [ ]:
joint = json.loads((ROOT / "data" / "joint_Vp_Q" / "summary.json").read_text())
best = min(joint["history"], key=lambda e: e["Q_rmse"]["whole"])
print("Best recorded synthetic Q RMSE:", best["epoch"], best["Q_rmse"])


## Inspect one checkpoint

Only epochs 10,20,...,100 have stored model arrays; all 100 epochs have scalar histories. Choose any saved epoch below.


In [ ]:
epoch = 60
initial = np.load(ROOT / "data" / "joint_Vp_Q" / "models_initial.npz", allow_pickle=False)
model = np.load(ROOT / "data" / "joint_Vp_Q" / f"models_epoch_{epoch:03d}.npz", allow_pickle=False)
fig, ax = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
for a, image, title in zip(ax, [initial["q_true"], initial["q_init"], model["Q"]], ["True Q", "Initial Q", f"Q epoch {epoch}"]):
    im = a.imshow(image, vmin=5, vmax=663, cmap="jet", aspect="auto", extent=[0,16.98,3.48,0])
    a.set(title=title, xlabel="Distance (km)", ylabel="Depth (km)")
fig.colorbar(im, ax=ax, label="Q (fixed water Q=1000 exceeds color scale)")
plt.show()


## Reproduction

See README.md and marmousi2_sls_fwi.py for explicit GPU rerun commands. The experiment workflow contains no solver implementation. It needs a compatible StarWave SLS package and suitable CUDA hardware. Defaults are five epochs; --epochs 100 reproduces the configured iteration count. This viewer intentionally does not launch that work.
